In [0]:
%pip install pypdf langchain langchain-text-splitters
dbutils.library.restartPython()

In [0]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

In [0]:
pdf_path = "/Volumes/dbr_dev_trial/artemzharkov10_bronze/raw_data/RAG_Data.pdf"
target_table = "dbr_dev_trial.artemzharkov10_gold.knowledge_base_chunks"

In [0]:
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text

document_text = extract_text_from_pdf(pdf_path)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,      
    chunk_overlap=30,    
    length_function=len,
)
chunks = text_splitter.create_documents([document_text])

In [0]:
data = []
for i, chunk in enumerate(chunks):
    data.append({
        "chunk_id": f"rag_data_chunk_{i}",
        "text_content": chunk.page_content,
        "source_file": "RAG_Data.pdf",
        "category": "road_safety"
    })

In [0]:
df = spark.createDataFrame(pd.DataFrame(data))

(df.write.format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(target_table))

In [0]:
# display(spark.sql(f"SELECT * FROM {target_table} LIMIT 5"))